# 06 · Grafo da memória por assunto

`graph.data(owner)` devolve o grafo em JSON (assuntos, memórias, ligações) e
`graph.html(owner, path)` gera uma página única e interativa que abre sem internet: arrastar,
aproximar (roda do mouse ou dois dedos), filtrar por assunto, buscar texto e clicar numa
memória para ver o texto e a mais parecida.

> **Instalação** (uma vez):
> ```bash
> pip install "pi-memory[local]"
> ollama pull qwen2.5:14b
> ```
> Todos os dados deste notebook são fictícios. As saídas abaixo vêm de uma execução real.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso do tqdm no Jupyter

def mostrar(obj, limite=1500):
    """JSON legível (o mesmo formato da biblioteca, do HTTP e do MCP)."""
    texto = json.dumps(obj, ensure_ascii=False, indent=2)
    print(texto if len(texto) <= limite else texto[:limite] + "\n…")

def pasta_limpa(nome):
    """Cada notebook começa do zero: apaga a pasta de dados da execução anterior."""
    shutil.rmtree(nome, ignore_errors=True)
    return nome

In [2]:
from pi_memory import PiMemory
pm = PiMemory(path=pasta_limpa("dados_06"), llm=lambda p: "")
for texto, assunto in [
    ("O usuário trabalha como enfermeira", "trabalho"),
    ("O usuário faz plantão noturno às quartas", "trabalho"),
    ("O usuário lidera a equipe de triagem", "trabalho"),
    ("O usuário tem uma filha chamada Lia", "família"),
    ("O usuário tem um filho chamado Davi", "família"),
    ("O usuário corre 5 km toda manhã", "saúde"),
    ("O usuário é alérgico a camarão", "saúde"),
    ("O usuário gosta de jazz", "lazer"),
]:
    pm.memories.remember(texto, owner="ana", topic=assunto)

## Os dados

In [3]:
grafo = pm.graph.data("ana")
print(len(grafo["nodes"]), "nós,", len(grafo["edges"]), "ligações")
mostrar(grafo["nodes"][:3])
mostrar([e for e in grafo["edges"] if e["kind"] == "similar"][:3])

12 nós, 12 ligações
[
  {
    "id": "t:família",
    "kind": "topic",
    "label": "Família",
    "topic": "família"
  },
  {
    "id": "m:3973e817-d5af-459c-b255-6b703c838574",
    "kind": "memory",
    "label": "O usuário tem uma filha chamada Lia",
    "topic": "família",
    "text": "O usuário tem uma filha chamada Lia"
  },
  {
    "id": "m:aa2c5fb7-0908-4108-9d29-7152d5a2e682",
    "kind": "memory",
    "label": "O usuário tem um filho chamado Davi",
    "topic": "família",
    "text": "O usuário tem um filho chamado Davi"
  }
]
[
  {
    "source": "m:3973e817-d5af-459c-b255-6b703c838574",
    "target": "m:aa2c5fb7-0908-4108-9d29-7152d5a2e682",
    "kind": "similar",
    "weight": 0.9384
  },
  {
    "source": "m:90f9f408-1386-4ee1-87d1-3bd5e52ff317",
    "target": "m:0e04fc8b-83d9-4542-a55e-aa79efc086a4",
    "kind": "similar",
    "weight": 0.8988
  },
  {
    "source": "m:888d37c7-5a67-4143-850f-6dd1b2fb6705",
    "target": "m:9d5fa00b-4aad-4112-9f7b-4f568802b057",
    "kind":

## A página

In [4]:
print(pm.graph.html("ana", path="dados_06/grafo.html"))
pm.close()
# Abra o arquivo no navegador. Num Jupyter local também dá para ver aqui:
# from IPython.display import IFrame; IFrame("dados_06/grafo.html", width="100%", height=500)

dados_06\grafo.html


Pela linha de comando:

```bash
pi-memory --path dados_06 graph html --owner ana --out grafo.html
```

![Grafo com 40 memórias fictícias](grafo_exemplo.png)